# Prodigy InfoTech - Data Science Internship
## Task 4: Sentiment Analysis on Social Media Data

**Goal:** Analyze and visualize sentiment patterns in social media data (Twitter) to understand public opinion towards specific brands/entities.

**Dataset:** `twitter_training.csv` (pre-labeled sentiment data containing ID, Entity/Topic, Sentiment, and Tweet text).

### 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# Viz styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

### 2. Load dataset

In [ ]:
# No header in the raw file, so we define columns manually
df = pd.read_csv("twitter_training.csv", header=None, names=["id", "entity", "sentiment", "text"])
print(df.shape)
df.head()

### 3. Data Cleaning
- Drop rows with missing text values
- Remove duplicate tweets based on ID + text to keep metrics clean
- Calculate length properties (chars, words)

In [ ]:
df = df.dropna(subset=["text"])
df = df.drop_duplicates(subset=["id", "text"])

df["char_length"] = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

print("Remaining rows:", df.shape[0])

### 4. Exploratory Data Analysis (EDA)

#### A. Overall Sentiment Count

In [ ]:
sentiment_counts = df["sentiment"].value_counts()
print(sentiment_counts)

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="sentiment", order=sentiment_counts.index, palette="viridis")
plt.title("Overall Sentiment Distribution")
plt.show()

#### B. Top 15 Most Discussed Brands/Entities

In [ ]:
entity_counts = df["entity"].value_counts().head(15)
sns.barplot(x=entity_counts.values, y=entity_counts.index, palette="mako")
plt.title("Top 15 Active Brands/Entities in Dataset")
plt.xlabel("Tweet Volume")
plt.show()

#### C. Sentiment Distribution per Brand
Normalized comparison to see positive vs negative proportions.

In [ ]:
brand_sentiment = pd.crosstab(df["entity"], df["sentiment"], normalize="index") * 100
brand_sentiment = brand_sentiment.sort_values(by="Negative", ascending=False).head(12)

brand_sentiment.plot(kind="barh", stacked=True, color=["#d9534f", "#5bc0de", "#f0ad4e", "#5cb85c"], figsize=(10, 6))
plt.title("Sentiment Split for Top 12 Brands (Sorted by Negative Ratio)")
plt.xlabel("%")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

#### D. Tweet Length by Sentiment

In [ ]:
print(df.groupby("sentiment")[["char_length", "word_count"]].mean().round(1))

sns.boxplot(data=df[df["char_length"] <= 300], x="sentiment", y="char_length", palette="Set2")
plt.title("Tweet Character Length by Sentiment")
plt.show()

### 5. Common Words Analysis
Excluding general stopwords to find topics discussed.

In [ ]:
STOPWORDS = {"the", "to", "and", "a", "of", "in", "is", "for", "on", "that", "this", "it", "with", "i", "you", "my", "me", "im", "game", "get", "dont", "like", "cant", "play"}

def get_tokens(text):
    if not isinstance(text, str):
        return []
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^\w\s]", "", text).lower()
    return [w for w in text.split() if w not in STOPWORDS and len(w) > 2]

pos_words = []
neg_words = []

for t in df[df["sentiment"] == "Positive"]["text"]:
    pos_words.extend(get_tokens(t))
for t in df[df["sentiment"] == "Negative"]["text"]:
    neg_words.extend(get_tokens(t))

print("Positive Top 8 Words:", Counter(pos_words).most_common(8))
print("Negative Top 8 Words:", Counter(neg_words).most_common(8))

### 6. Key Takeaways
- Sentiment proportions are fairly distributed, with a slight negative lean overall.
- Different entities (such as competitive games or service providers) have markedly different positive/negative ratios.
- Neutral tweets are generally shorter, while emotionally charged tweets (both positive and negative) show longer average word counts.